Import Library

Loughran-Mcdonald Sentiment Extractor

In [1]:
class LMSentimentExtractor:
    def __init__(self, csv_path = None):
        self.lexicon = self._load_lexicon(csv_path)
        self.negations = {'no', 'not', 'none', 'neither', 'never', 'nobody'}
    
    def _load_lexicon(self, csv_path):
        lexicon = {'positive': set(), 'negative': set(), 'uncertain': set(), 'litigious': set()}
        df = pd.read_csv("C:/Users/tnk20/nlp/Loughran-McDonald_MasterDictionary_1993-2024.csv")
        lexicon['negative'] = set(df[df['Negative'] > 0]['Word'].str.lower())
        lexicon['positive'] = set(df[df['Positive'] > 0]['Word'].str.lower())
        lexicon['uncertain'] = set(df[df['Uncertainty'] > 0]['Word'].str.lower())
        lexicon['litigious'] = set(df[df['Litigious'] > 0]['Word'].str.lower())
        print(">> Loaded external LM Dictionary.")        

    def get_sentiment_features(self, text):
        tokens = nltk.word_tokenize(text.lower())
        total_tokens = len(tokens)
        if total_tokens == 0:
            return [0.0, 0.0, 0.0, 0.0]

        counts = {'positive': 0, 'negative': 0, 'uncertain': 0, 'litigious': 0}
            
        for i, token in enumerate(tokens):
            # Check for negation in the preceding 3 words 
            is_negated = False
            start_window = max(0, i - 3)
            if any(t in self.negations for t in tokens[start_window:i]):
                is_negated = True

            # Logic: If negated, we might flip 'positive' to 'negative' or ignore.
            # Standard LM approach is often to ignore positive words if negated.
            
            if token in self.lexicon['positive']:
                if not is_negated:
                    counts['positive'] += 1
                # If negated positive (e.g. "not good"), some implementations count as negative
                # strictly following the paper's simple "counts" logic implies we just skip the positive count.
            
            elif token in self.lexicon['negative']:
                # Negated negative (e.g. "no recession") is usually treated as neutral (ignored)
                if not is_negated:
                    counts['negative'] += 1
            
            elif token in self.lexicon['uncertain']:
                counts['uncertain'] += 1
            
            elif token in self.lexicon['litigious']:
                counts['litigious'] += 1

        # Normalize by document length 
        return [
            counts['positive'] / total_tokens,
            counts['negative'] / total_tokens,
            counts['uncertain'] / total_tokens,
            counts['litigious'] / total_tokens
        ]

Data Processor and Feature Engineering

In [2]:
class FedDataProcessor:
    def __init__(self):
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=CONFIG['tfidf_features'],
            stop_words='english',
            norm='l2'
        )
        self.lm_extractor = LMSentimentExtractor() # Pass csv_path='LoughranMcDonald_MasterDictionary.csv' here

    def preprocess_text(self, text):
        """
        Cleaning based on Section V.A [cite: 164-166]
        Note: Paper excludes lemmatization for domain-specific terms[cite: 166].
        """
        # Remove non-alpha chars, keep spaces
        text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        return text

    def process_pipeline(self, df_economic, df_text):
        print(">> Step 1: Text Preprocessing...")
        clean_texts = df_text['full_text'].apply(self.preprocess_text)

        print(">> Step 2: Generating TF-IDF Features (Method 1)...")
        # Creates T_i vector [cite: 264]
        tfidf_matrix = self.tfidf_vectorizer.fit_transform(clean_texts).toarray()
        tfidf_cols = [f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
        df_tfidf = pd.DataFrame(tfidf_matrix, columns=tfidf_cols, index=df_text.index)

        print(">> Step 3: Generating LM Sentiment Features (Method 1)...")
        # Creates L_i vector [cite: 310]
        lm_features = df_text['full_text'].apply(self.lm_extractor.get_sentiment_features)
        df_lm = pd.DataFrame(lm_features.tolist(), 
                             columns=['lm_pos', 'lm_neg', 'lm_unc', 'lm_lit'], 
                             index=df_text.index)

        print(">> Step 4: Merging Structured and Unstructured Data...")
        # Concatenate: F_i <- [E_i, T_i, L_i] [cite: 311]
        # Align by index (Date)
        full_df = pd.concat([df_economic, df_tfidf, df_lm], axis=1).dropna()
        
        return full_df

XGBoost Model Training

In [3]:
class FedPredictor:
    def __init__(self):
        # Gradient Boosting with params from Section IV.D 
        self.model = xgb.XGBClassifier(
            n_estimators=CONFIG['n_estimators'],
            learning_rate=CONFIG['learning_rate'],
            max_depth=CONFIG['max_depth'],
            min_child_weight=CONFIG['min_samples_leaf'],
            objective='multi:softprob', # For 3-class probability
            num_class=3,
            random_state=CONFIG['random_state'],
            eval_metric='mlogloss'
        )
        self.scaler = StandardScaler()

    def train_evaluate(self, X, y):
        """
        Stratified 5-Fold CV + SMOTE [cite: 136-137]
        """
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG['random_state'])
        auc_scores, acc_scores = [], []

        print(f">> Training XGBoost on {X.shape[1]} features (Hybrid Method 1)...")

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Standardize Features (Paper implies standardization of economic vars)
            X_train_scaled = self.scaler.fit_transform(X_train)
            X_test_scaled = self.scaler.transform(X_test)

            # Apply SMOTE to Training Data Only [cite: 147]
            smote = SMOTE(random_state=CONFIG['random_state'])
            X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

            # Train
            self.model.fit(X_res, y_res)

            # Predict
            y_pred = self.model.predict(X_test_scaled)
            y_proba = self.model.predict_proba(X_test_scaled)

            # Metrics
            # Multi-class AUC (OvR)
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
            acc = accuracy_score(y_test, y_pred)
            
            auc_scores.append(auc)
            acc_scores.append(acc)

        print("\n=== METHOD 1 RESULTS [cite: 154] ===")
        print(f"Mean Test AUC: {np.mean(auc_scores):.4f}")
        print(f"Mean Test Accuracy: {np.mean(acc_scores):.4f}")
        return self.model

Collect Data

In [ ]:
#Structured data
filename_ed = ['CPIAUCSL', 'HOUST', 'HPI', 'NFP', 'PCE', 'T10Y3MM', 'UNRATE']
df_ed = read.csv